# S6.2 — Guardrail Node

Domain relevance check (0-100) using structured LLM output.
First node in the agentic RAG workflow — rejects off-topic queries before expensive retrieval.

In [1]:
import sys
sys.path.insert(0, "../..")

from src.services.agents.nodes.guardrail_node import (
    GUARDRAIL_PROMPT,
    ainvoke_guardrail_step,
    continue_after_guardrail,
    get_latest_query,
)
from src.services.agents.state import AgentState, create_initial_state
from src.services.agents.context import AgentContext
from src.services.agents.models import GuardrailScoring

print("\u2713 All imports successful")

✓ All imports successful


## FR-3: Query Extraction Helper

In [2]:
from langchain_core.messages import HumanMessage, AIMessage

# Single message
msgs = [HumanMessage(content="How does BERT work?")]
assert get_latest_query(msgs) == "How does BERT work?"

# Multiple messages — extracts last HumanMessage
msgs = [
    HumanMessage(content="First question"),
    AIMessage(content="Answer"),
    HumanMessage(content="Follow-up"),
]
assert get_latest_query(msgs) == "Follow-up"

# Empty list raises
try:
    get_latest_query([])
    assert False, "Should have raised"
except ValueError:
    pass

print("\u2713 get_latest_query works correctly")

✓ get_latest_query works correctly


## FR-4: Guardrail Prompt

In [3]:
# Prompt exists and is formattable
assert isinstance(GUARDRAIL_PROMPT, str)
assert "{question}" in GUARDRAIL_PROMPT

formatted = GUARDRAIL_PROMPT.format(question="How does BERT work?")
assert "How does BERT work?" in formatted
assert "research" in GUARDRAIL_PROMPT.lower() or "academic" in GUARDRAIL_PROMPT.lower()

print("Prompt preview (first 200 chars):")
print(GUARDRAIL_PROMPT[:200])
print("\n\u2713 GUARDRAIL_PROMPT is valid and formattable")

Prompt preview (first 200 chars):
You are a domain relevance classifier for an academic research assistant.

Rate the following question on a scale of 0-100 for relevance to academic and scientific research,
particularly in the areas 

✓ GUARDRAIL_PROMPT is valid and formattable


## FR-2: Conditional Edge Routing

In [4]:
from unittest.mock import MagicMock

mock_provider = MagicMock()
ctx = AgentContext(llm_provider=mock_provider, guardrail_threshold=40)

# Above threshold → continue
state = AgentState(guardrail_result=GuardrailScoring(score=85, reason="Academic"))
assert continue_after_guardrail(state, ctx) == "continue"

# Below threshold → out_of_scope
state = AgentState(guardrail_result=GuardrailScoring(score=10, reason="Off-topic"))
assert continue_after_guardrail(state, ctx) == "out_of_scope"

# Exact threshold → continue
state = AgentState(guardrail_result=GuardrailScoring(score=40, reason="Borderline"))
assert continue_after_guardrail(state, ctx) == "continue"

# No result → defaults to continue
state = AgentState()
assert continue_after_guardrail(state, ctx) == "continue"

print("\u2713 continue_after_guardrail routing logic correct")

No guardrail_result in state - defaulting to continue


✓ continue_after_guardrail routing logic correct


## FR-1: Guardrail Scoring Node (mocked LLM)

In [5]:
import asyncio
from unittest.mock import AsyncMock, MagicMock

# Setup mock LLM provider
mock_provider = MagicMock()
mock_model = MagicMock()
mock_structured = AsyncMock()
mock_structured.ainvoke = AsyncMock(
    return_value=GuardrailScoring(score=88, reason="BERT is a core NLP model")
)
mock_model.with_structured_output.return_value = mock_structured
mock_provider.get_langchain_model.return_value = mock_model

ctx = AgentContext(llm_provider=mock_provider, model_name="test-model")
state = create_initial_state("How does BERT work?")

result = await ainvoke_guardrail_step(state, ctx)

assert "guardrail_result" in result
assert result["guardrail_result"].score == 88
assert result["guardrail_result"].reason == "BERT is a core NLP model"

# Verify temperature=0.0 was used
call_kwargs = mock_provider.get_langchain_model.call_args
assert call_kwargs.kwargs.get("temperature") == 0.0

# Verify structured output
mock_model.with_structured_output.assert_called_with(GuardrailScoring)

print(f"Score: {result['guardrail_result'].score}")
print(f"Reason: {result['guardrail_result'].reason}")
print("\n\u2713 ainvoke_guardrail_step works with mocked LLM")

Score: 88
Reason: BERT is a core NLP model

✓ ainvoke_guardrail_step works with mocked LLM


## LLM Failure Fallback

In [6]:
# Setup mock LLM that fails
mock_provider = MagicMock()
mock_model = MagicMock()
mock_structured = AsyncMock()
mock_structured.ainvoke = AsyncMock(side_effect=RuntimeError("LLM timeout"))
mock_model.with_structured_output.return_value = mock_structured
mock_provider.get_langchain_model.return_value = mock_model

ctx = AgentContext(llm_provider=mock_provider, model_name="test-model")
state = create_initial_state("How does BERT work?")

result = await ainvoke_guardrail_step(state, ctx)

assert result["guardrail_result"].score == 50
assert "failed" in result["guardrail_result"].reason.lower() or "fallback" in result["guardrail_result"].reason.lower()

print(f"Fallback score: {result['guardrail_result'].score}")
print(f"Fallback reason: {result['guardrail_result'].reason}")
print("\n\u2713 Graceful fallback on LLM failure (score=50, above threshold=40)")

Guardrail LLM call failed: LLM timeout - falling back to score=50


Fallback score: 50
Fallback reason: LLM validation failed, using conservative fallback: LLM timeout

✓ Graceful fallback on LLM failure (score=50, above threshold=40)


## End-to-End Flow (mocked)

In [7]:
# Simulate full guardrail flow: score → route
mock_provider = MagicMock()
mock_model = MagicMock()

# On-topic query
mock_structured = AsyncMock()
mock_structured.ainvoke = AsyncMock(
    return_value=GuardrailScoring(score=92, reason="Transformer architecture is core research")
)
mock_model.with_structured_output.return_value = mock_structured
mock_provider.get_langchain_model.return_value = mock_model

ctx = AgentContext(llm_provider=mock_provider, model_name="test-model", guardrail_threshold=40)
state = create_initial_state("How do transformers work in NLP?")

# Step 1: Score
result = await ainvoke_guardrail_step(state, ctx)
state_with_result = AgentState(guardrail_result=result["guardrail_result"])

# Step 2: Route
route = continue_after_guardrail(state_with_result, ctx)
assert route == "continue"
print(f"On-topic: score={result['guardrail_result'].score} → route='{route}'")

# Off-topic query
mock_structured.ainvoke = AsyncMock(
    return_value=GuardrailScoring(score=5, reason="Pizza is not research")
)
state = create_initial_state("What is the best pizza in NYC?")

result = await ainvoke_guardrail_step(state, ctx)
state_with_result = AgentState(guardrail_result=result["guardrail_result"])
route = continue_after_guardrail(state_with_result, ctx)
assert route == "out_of_scope"
print(f"Off-topic: score={result['guardrail_result'].score} → route='{route}'")

print("\n\u2713 End-to-end guardrail flow works correctly")

On-topic: score=92 → route='continue'
Off-topic: score=5 → route='out_of_scope'

✓ End-to-end guardrail flow works correctly
